In [4]:
import requests
import pandas as pd
from typing import List, Dict
import json

# SEC EDGAR API doesn't require an API key for basic access
class SECEdgarAPI:
    def __init__(self):
        self.base_url = "https://data.sec.gov/api/xbrl"
        self.submissions_url = "https://data.sec.gov/submissions"
        
    def get_company_filings(self, ticker: str, filing_type: str = "10-K") -> List[Dict]:
        """
        Get company filings by ticker symbol
        """
        # First, get the CIK (Central Index Key) for the company
        cik = self._get_cik_from_ticker(ticker)
        if not cik:
            print(f"Could not find CIK for ticker: {ticker}")
            return []
        
        # Get submissions data
        submissions_url = f"{self.submissions_url}/CIK{cik.zfill(10)}.json"
        headers = {
            'User-Agent': 'Your Company Name your.email@domain.com',
            'Accept-Encoding': 'gzip, deflate'
        }
        
        try:
            response = requests.get(submissions_url, headers=headers)
            response.raise_for_status()
            data = response.json()
            
            # Extract filings
            filings = data.get('filings', {}).get('recent', {})
            filing_data = []
            
            # Match filing types
            for i in range(len(filings.get('form', []))):
                if filings['form'][i] == filing_type:
                    filing_data.append({
                        'accession_number': filings['accessionNumber'][i],
                        'filing_date': filings['filingDate'][i],
                        'report_date': filings['reportDate'][i],
                        'primary_document': filings['primaryDocument'][i],
                        'form': filings['form'][i]
                    })
            
            return filing_data
            
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            return []
    
    def _get_cik_from_ticker(self, ticker: str) -> str:
        """
        Convert ticker symbol to CIK number
        """
        # You can use a local mapping or fetch from SEC
        ticker_to_cik = {
            'AAPL': '0000320193',
            'MSFT': '0000789019',
            'GOOGL': '0001652044',
            'AMZN': '0001018724',
            'TSLA': '0001318605'
        }
        
        return ticker_to_cik.get(ticker.upper(), '')

# Usage example
sec_api = SECEdgarAPI()
filings = sec_api.get_company_filings('AAPL', '10-K')
print(f"Found {len(filings)} 10-K filings for Apple")
for filing in filings[:3]:  # Show first 3
    print(f"Date: {filing['filing_date']}, Report: {filing['report_date']}")

Found 10 10-K filings for Apple
Date: 2024-11-01, Report: 2024-09-28
Date: 2023-11-03, Report: 2023-09-30
Date: 2022-10-28, Report: 2022-09-24


In [5]:
import os

# Create the 'filings' directory if it doesn't exist
os.makedirs('filings', exist_ok=True)

# Save each filing's content as a JSON file in the 'filings' folder
for filing in filings:
    filename = f"filings/{filing['accession_number']}.json"
    with open(filename, 'w') as f:
        json.dump(filing, f, indent=2)

In [6]:
def get_full_filing_content(metadata: dict, ticker: str):
    """
    Fetch the actual filing content using the metadata
    """
    cik = "0000320193"  # Apple's CIK
    accession_number = metadata['accession_number'].replace('-', '')
    document_name = metadata['primary_document']
    
    # Construct the URL to the actual filing
    filing_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_number}/{document_name}"
    
    headers = {
        'User-Agent': 'Your Company Name your.email@domain.com',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    try:
        response = requests.get(filing_url, headers=headers)
        response.raise_for_status()
        
        # This returns the actual HTML content of the 10-K filing
        return response.text
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching filing content: {e}")
        return None

# Usage - add this to your existing code
metadata = {
  "accession_number": "0000320193-24-000123",
  "filing_date": "2024-11-01", 
  "report_date": "2024-09-28",
  "primary_document": "aapl-20240928.htm",
  "form": "10-K"
}

# Get the actual filing content
filing_content = get_full_filing_content(metadata, 'AAPL')

if filing_content:
    print(f"Successfully retrieved filing with {len(filing_content)} characters")
    print("This contains the actual financial statements, MD&A, risk factors, etc.")
    
    # You can save it to a file to examine
    with open('apple_10k_2024.html', 'w', encoding='utf-8') as f:
        f.write(filing_content)
    print("Saved to apple_10k_2024.html")

Successfully retrieved filing with 1503780 characters
This contains the actual financial statements, MD&A, risk factors, etc.
Saved to apple_10k_2024.html


In [14]:
# sec_filings.py
from __future__ import annotations
import datetime as dt
import json
import re
from pathlib import Path
from typing import Iterable, List, Dict, Optional

import requests
from requests.adapters import HTTPAdapter, Retry

# ---------- Config ----------
SEC_HEADERS = {
    # Put YOUR email or site here so SEC can reach you if needed
    "User-Agent": "DueDiligenceBot/1.0 (surajonlyforgames@gmail.com)",
    "Accept": "application/json",
}
COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL_TMPL = "https://data.sec.gov/submissions/CIK{cik:010d}.json"

CACHE_DIR = Path(".sec_cache")
CACHE_DIR.mkdir(exist_ok=True)
TICKERS_CACHE = CACHE_DIR / "company_tickers.json"

# ---------- HTTP helper with polite retries ----------
def _session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=5, backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update(SEC_HEADERS)
    return s

# ---------- CIK resolution ----------
def load_company_index(use_cache: bool = True) -> List[Dict]:
    """
    Loads the SEC public-company index (ticker/name -> CIK).
    Returns a list of rows: {"cik": int, "ticker": "AAPL", "name": "APPLE INC"}
    """
    if use_cache and TICKERS_CACHE.exists():
        data = json.loads(TICKERS_CACHE.read_text())
    else:
        with _session().get(COMPANY_TICKERS_URL, timeout=30) as r:
            r.raise_for_status()
            data = r.json()
        TICKERS_CACHE.write_text(json.dumps(data))  # cache to disk

    rows = []
    for _, row in data.items():
        rows.append({
            "cik": int(row["cik_str"]),
            "ticker": str(row.get("ticker", "")).upper(),
            "name": str(row.get("title", "")).upper(),
        })
    return rows

def resolve_cik(query: str, company_index: List[Dict]) -> Optional[int]:
    """
    Resolve user query (ticker like 'AAPL' or company name like 'Apple') to a CIK.
    Strategy: exact ticker match -> name contains -> startswith -> fuzzy token match.
    """
    q = query.strip().upper()

    # exact ticker hit
    for row in company_index:
        if row["ticker"] and row["ticker"] == q:
            return row["cik"]

    # name contains
    contains = [r for r in company_index if q in r["name"]]
    if contains:
        return contains[0]["cik"]

    # name startswith
    starts = [r for r in company_index if r["name"].startswith(q)]
    if starts:
        return starts[0]["cik"]

    # token match (e.g., "ALPHABET INC CLASS A" vs "ALPHABET")
    tokens = set(re.findall(r"[A-Z0-9]+", q))
    best = None
    for r in company_index:
        name_tokens = set(r["name"].split())
        if tokens.issubset(name_tokens):
            best = r["cik"]
            break
    return best

# ---------- Submissions fetch & filter ----------
def fetch_submissions(cik: int) -> Dict:
    """
    Get the company's submissions JSON (recent filings + pointers to older shards).
    """
    url = SUBMISSIONS_URL_TMPL.format(cik=cik)
    with _session().get(url, timeout=30) as r:
        r.raise_for_status()
        return r.json()

def _iterate_recent(fj: Dict):
    """
    Yield row dicts from filings.recent (columnar arrays to row dicts).
    """
    recent = fj.get("filings", {}).get("recent", {})
    cols = ["form","filingDate","reportDate","accessionNumber","primaryDocument"]
    arrays = [recent.get(k, []) for k in cols]
    for i in range(min(len(a) for a in arrays)):
        yield {k: arrays[j][i] for j, k in enumerate(cols)}

def build_primary_doc_url(cik: int | str, accession_no: str, primary_doc: str) -> str:
    """
    Canonical EDGAR Archives path for the primary document in a submission.
    https://www.sec.gov/Archives/edgar/data/{CIK_no_leading_zeros}/{accessionNo_no_dashes}/{primaryDocument}
    """
    cik_no_zeros = str(int(cik))
    acc_nodash = accession_no.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_no_zeros}/{acc_nodash}/{primary_doc}"

def filings_in_range(
    filings_json: Dict,
    forms: Iterable[str],
    start: dt.date,
    end: dt.date,
) -> List[Dict]:
    """
    Filter recent filings by form type and filingDate within [start, end].
    Returns list of dicts with URL to the primary document.
    """
    hits = []
    for row in _iterate_recent(filings_json):
        if row["form"] not in forms:
            continue
        fdate = dt.date.fromisoformat(row["filingDate"])
        if not (start <= fdate <= end):
            continue
        url = build_primary_doc_url(filings_json["cik"], row["accessionNumber"], row["primaryDocument"])
        hits.append({
            "form": row["form"],
            "filingDate": row["filingDate"],
            "reportDate": row.get("reportDate", ""),
            "accessionNumber": row["accessionNumber"],
            "primaryDocument": row["primaryDocument"],
            "url": url,
        })
    # Sort newest first
    hits.sort(key=lambda d: d["filingDate"], reverse=True)
    return hits

def find_10k_10q(
    company_or_ticker: str,
    year: Optional[int] = None,
    start: Optional[str] = None,  # "YYYY-MM-DD"
    end: Optional[str]   = None,
) -> List[Dict]:
    """
    Main entry point: Return 10-K and 10-Q primary document URLs for a company
    and either a single year or a closed date range [start, end].
    """
    idx = load_company_index()
    cik = resolve_cik(company_or_ticker, idx)
    if cik is None:
        raise ValueError(f"Could not resolve CIK for '{company_or_ticker}'. Try exact ticker or official EDGAR name.")

    subs = fetch_submissions(cik)

    if year is not None:
        start_date = dt.date(year, 1, 1)
        end_date   = dt.date(year, 12, 31)
    else:
        if not start or not end:
            raise ValueError("Provide either 'year' OR both 'start' and 'end' (YYYY-MM-DD).")
        start_date = dt.date.fromisoformat(start)
        end_date   = dt.date.fromisoformat(end)

    return filings_in_range(subs, {"10-K", "10-Q"}, start_date, end_date)

if __name__ == "__main__":
    # Quick manual test:
    results = find_10k_10q("META", year=2025)
    for r in results:
        print(r["form"], r["filingDate"], "->", r["url"])


10-Q 2025-07-31 -> https://www.sec.gov/Archives/edgar/data/1326801/000162828025036791/meta-20250630.htm
10-Q 2025-05-01 -> https://www.sec.gov/Archives/edgar/data/1326801/000132680125000054/meta-20250331.htm
10-K 2025-01-30 -> https://www.sec.gov/Archives/edgar/data/1326801/000132680125000017/meta-20241231.htm


In [11]:
# sec_filings.py
from __future__ import annotations
import datetime as dt
import json
import re
import time
from pathlib import Path
from typing import Iterable, List, Dict, Optional

import requests
from requests.adapters import HTTPAdapter, Retry

# ---------- Config ----------
SEC_HEADERS = {
    # Put YOUR email or site here so SEC can reach you if needed
    "User-Agent": "DueDiligenceBot/1.0 (surajonlyforgames@gmail.com)",
    "Accept": "application/json",
}
COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL_TMPL = "https://data.sec.gov/submissions/CIK{cik:010d}.json"

CACHE_DIR = Path(".sec_cache")
CACHE_DIR.mkdir(exist_ok=True)
TICKERS_CACHE = CACHE_DIR / "company_tickers.json"

# ---------- HTTP helper with polite retries ----------
def _session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=5, 
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update(SEC_HEADERS)
    return s

# ---------- CIK resolution ----------
def load_company_index(use_cache: bool = True) -> List[Dict]:
    """
    Loads the SEC public-company index (ticker/name -> CIK).
    Returns a list of rows: {"cik": int, "ticker": "AAPL", "name": "APPLE INC"}
    """
    if use_cache and TICKERS_CACHE.exists():
        data = json.loads(TICKERS_CACHE.read_text())
    else:
        with _session().get(COMPANY_TICKERS_URL, timeout=30) as r:
            r.raise_for_status()
            data = r.json()
        TICKERS_CACHE.write_text(json.dumps(data))  # cache to disk

    rows = []
    for _, row in data.items():
        rows.append({
            "cik": int(row["cik_str"]),
            "ticker": str(row.get("ticker", "")).upper(),
            "name": str(row.get("title", "")).upper(),
        })
    return rows

def resolve_cik(query: str, company_index: List[Dict]) -> Optional[int]:
    """
    Resolve user query (ticker like 'AAPL' or company name like 'Apple') to a CIK.
    Strategy: exact ticker match -> name contains -> startswith -> fuzzy token match.
    """
    q = query.strip().upper()
    
    print(f"🔍 Searching for: '{q}'")

    # exact ticker hit
    for row in company_index:
        if row["ticker"] and row["ticker"] == q:
            print(f"✅ Exact ticker match: {row['ticker']} -> {row['name']} (CIK: {row['cik']})")
            return row["cik"]

    # name exact match
    for row in company_index:
        if row["name"] == q:
            print(f"✅ Exact name match: {row['name']} (CIK: {row['cik']})")
            return row["cik"]

    # name contains
    contains = [r for r in company_index if q in r["name"]]
    if contains:
        print(f"✅ Name contains match: {contains[0]['name']} (CIK: {contains[0]['cik']})")
        return contains[0]["cik"]

    # name startswith
    starts = [r for r in company_index if r["name"].startswith(q)]
    if starts:
        print(f"✅ Name starts with match: {starts[0]['name']} (CIK: {starts[0]['cik']})")
        return starts[0]["cik"]

    # token match (e.g., "ALPHABET INC CLASS A" vs "ALPHABET")
    tokens = set(re.findall(r"[A-Z0-9]+", q))
    best = None
    best_score = 0
    for r in company_index:
        name_tokens = set(r["name"].split())
        common_tokens = tokens.intersection(name_tokens)
        score = len(common_tokens)
        if score > best_score:
            best_score = score
            best = r["cik"]
    
    if best:
        company_name = next(r["name"] for r in company_index if r["cik"] == best)
        print(f"✅ Token match: {company_name} (CIK: {best})")
        return best

    print(f"❌ No match found for '{q}'")
    return None

# ---------- Submissions fetch & filter ----------
def fetch_submissions(cik: int) -> Dict:
    """
    Get the company's submissions JSON (recent filings + pointers to older shards).
    """
    url = SUBMISSIONS_URL_TMPL.format(cik=cik)
    print(f"📥 Fetching submissions from: {url}")
    
    with _session().get(url, timeout=30) as r:
        r.raise_for_status()
        data = r.json()
        
    # Debug: print available forms and recent filing counts
    recent_filings = data.get("filings", {}).get("recent", {})
    forms = recent_filings.get("form", [])
    filing_dates = recent_filings.get("filingDate", [])
    
    print(f"📊 Found {len(forms)} recent filings")
    form_counts = {}
    for form in forms:
        form_counts[form] = form_counts.get(form, 0) + 1
    
    for form, count in form_counts.items():
        print(f"   - {form}: {count} filings")
    
    return data

def _iterate_recent(fj: Dict):
    """
    Yield row dicts from filings.recent (columnar arrays to row dicts).
    """
    recent = fj.get("filings", {}).get("recent", {})
    cols = ["form","filingDate","reportDate","accessionNumber","primaryDocument"]
    arrays = [recent.get(k, []) for k in cols]
    
    if not arrays or min(len(a) for a in arrays) == 0:
        return
    
    for i in range(min(len(a) for a in arrays)):
        yield {k: arrays[j][i] for j, k in enumerate(cols)}

def build_primary_doc_url(cik: int | str, accession_no: str, primary_doc: str) -> str:
    """
    Canonical EDGAR Archives path for the primary document in a submission.
    https://www.sec.gov/Archives/edgar/data/{CIK_no_leading_zeros}/{accessionNo_no_dashes}/{primaryDocument}
    """
    cik_no_zeros = str(int(cik))
    acc_nodash = accession_no.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_no_zeros}/{acc_nodash}/{primary_doc}"

def filings_in_range(
    filings_json: Dict,
    forms: Iterable[str],
    start: dt.date,
    end: dt.date,
) -> List[Dict]:
    """
    Filter recent filings by form type and filingDate within [start, end].
    Returns list of dicts with URL to the primary document.
    """
    hits = []
    forms_set = set(forms)
    
    print(f"🔎 Filtering for forms {forms_set} between {start} and {end}")
    
    for row in _iterate_recent(filings_json):
        if row["form"] not in forms_set:
            continue
        try:
            fdate = dt.date.fromisoformat(row["filingDate"])
            if not (start <= fdate <= end):
                continue
                
            url = build_primary_doc_url(filings_json["cik"], row["accessionNumber"], row["primaryDocument"])
            hits.append({
                "form": row["form"],
                "filingDate": row["filingDate"],
                "reportDate": row.get("reportDate", ""),
                "accessionNumber": row["accessionNumber"],
                "primaryDocument": row["primaryDocument"],
                "url": url,
            })
        except (ValueError, KeyError) as e:
            print(f"⚠️  Skipping filing due to error: {e} - {row}")
            continue
    
    print(f"✅ Found {len(hits)} matching filings")
    # Sort newest first
    hits.sort(key=lambda d: d["filingDate"], reverse=True)
    return hits

def find_10k_10q(
    company_or_ticker: str,
    year: Optional[int] = None,
    start: Optional[str] = None,  # "YYYY-MM-DD"
    end: Optional[str]   = None,
    debug: bool = False,
) -> List[Dict]:
    """
    Main entry point: Return 10-K and 10-Q primary document URLs for a company
    and either a single year or a closed date range [start, end].
    
    Args:
        company_or_ticker: Company name or ticker symbol
        year: Single year to search (e.g., 2023)
        start: Start date in "YYYY-MM-DD" format
        end: End date in "YYYY-MM-DD" format  
        debug: Print debug information
    """
    print(f"🚀 Starting SEC filings search for: {company_or_ticker}")
    
    idx = load_company_index()
    cik = resolve_cik(company_or_ticker, idx)
    if cik is None:
        raise ValueError(f"Could not resolve CIK for '{company_or_ticker}'. Try exact ticker or official EDGAR name.")

    # Be polite to SEC servers
    time.sleep(0.5)
    
    try:
        subs = fetch_submissions(cik)
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            raise ValueError(f"CIK {cik} not found in SEC database. Company may be private or delisted.")
        else:
            raise

    if year is not None:
        start_date = dt.date(year, 1, 1)
        end_date   = dt.date(year, 12, 31)
        print(f"📅 Searching year: {year}")
    else:
        if not start or not end:
            raise ValueError("Provide either 'year' OR both 'start' and 'end' (YYYY-MM-DD).")
        start_date = dt.date.fromisoformat(start)
        end_date   = dt.date.fromisoformat(end)
        print(f"📅 Searching range: {start} to {end}")

    results = filings_in_range(subs, {"10-K", "10-Q"}, start_date, end_date)
    
    if not results:
        print("❌ No filings found. Possible reasons:")
        print("   - Company may not have filed 10-K/10-Q in specified period")
        print("   - Company may be private or foreign (filing different forms)")
        print("   - Data may be available in older filings (not in 'recent')")
        print("   - Try expanding your date range")
    
    return results

# Alternative search function for broader form types
def find_filings_broad(
    company_or_ticker: str,
    forms: List[str] = None,
    year: Optional[int] = None,
    start: Optional[str] = None,
    end: Optional[str] = None,
) -> List[Dict]:
    """
    Broader search that includes more form types and provides more debugging info.
    """
    if forms is None:
        forms = ["10-K", "10-Q", "8-K", "20-F", "6-K", "S-1"]
    
    print(f"🔍 Broad search for forms: {forms}")
    
    idx = load_company_index()
    cik = resolve_cik(company_or_ticker, idx)
    if cik is None:
        raise ValueError(f"Could not resolve CIK for '{company_or_ticker}'")

    time.sleep(0.5)
    subs = fetch_submissions(cik)

    if year is not None:
        start_date = dt.date(year, 1, 1)
        end_date   = dt.date(year, 12, 31)
    else:
        if not start or not end:
            raise ValueError("Provide either 'year' OR both 'start' and 'end' (YYYY-MM-DD).")
        start_date = dt.date.fromisoformat(start)
        end_date   = dt.date.fromisoformat(end)

    return filings_in_range(subs, forms, start_date, end_date)

if __name__ == "__main__":
    # Enhanced test with error handling
    test_cases = [
        "META",  # Should work
        "AAPL",  # Should work  
        "GOOGL", # Should work
        "TSLA",  # Should work
    ]
    
    for test in test_cases:
        try:
            print(f"\n{'='*50}")
            print(f"Testing: {test}")
            print(f"{'='*50}")
            results = find_10k_10q(test, year=2022)
            for r in results:
                print(f"📄 {r['form']} {r['filingDate']} -> {r['url']}")
        except Exception as e:
            print(f"❌ Error with {test}: {e}")


Testing: META
🚀 Starting SEC filings search for: META
🔍 Searching for: 'META'
✅ Exact ticker match: META -> META PLATFORMS, INC. (CIK: 1326801)
📥 Fetching submissions from: https://data.sec.gov/submissions/CIK0001326801.json
📊 Found 1001 recent filings
   - 144: 385 filings
   - 4: 543 filings
   - 10-Q: 5 filings
   - 8-K: 15 filings
   - S-8 POS: 8 filings
   - SD: 2 filings
   - PX14A6G: 19 filings
   - 3: 7 filings
   - ARS: 2 filings
   - DEFA14A: 2 filings
   - DEF 14A: 2 filings
   - SCHEDULE 13G/A: 1 filings
   - 10-K: 2 filings
   - 424B2: 2 filings
   - FWP: 1 filings
   - PRE 14A: 1 filings
   - SC 13G/A: 4 filings
📅 Searching year: 2022
🔎 Filtering for forms {'10-Q', '10-K'} between 2022-01-01 and 2022-12-31
✅ Found 0 matching filings
❌ No filings found. Possible reasons:
   - Company may not have filed 10-K/10-Q in specified period
   - Company may be private or foreign (filing different forms)
   - Data may be available in older filings (not in 'recent')
   - Try expandi

In [12]:
# sec_filings.py
from __future__ import annotations
import datetime as dt
import json
import re
import time
from pathlib import Path
from typing import Iterable, List, Dict, Optional

import requests
from requests.adapters import HTTPAdapter, Retry

# ---------- Config ----------
SEC_HEADERS = {
    "User-Agent": "DueDiligenceBot/1.0 (surajonlyforgames@gmail.com)",
    "Accept": "application/json",
}
COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL_TMPL = "https://data.sec.gov/submissions/CIK{cik:010d}.json"

CACHE_DIR = Path(".sec_cache")
CACHE_DIR.mkdir(exist_ok=True)
TICKERS_CACHE = CACHE_DIR / "company_tickers.json"

# ---------- HTTP helper with polite retries ----------
def _session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=5, 
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update(SEC_HEADERS)
    return s

# ---------- CIK resolution ----------
def load_company_index(use_cache: bool = True) -> List[Dict]:
    """Loads the SEC public-company index (ticker/name -> CIK)."""
    if use_cache and TICKERS_CACHE.exists():
        data = json.loads(TICKERS_CACHE.read_text())
    else:
        with _session().get(COMPANY_TICKERS_URL, timeout=30) as r:
            r.raise_for_status()
            data = r.json()
        TICKERS_CACHE.write_text(json.dumps(data))

    rows = []
    for _, row in data.items():
        rows.append({
            "cik": int(row["cik_str"]),
            "ticker": str(row.get("ticker", "")).upper(),
            "name": str(row.get("title", "")).upper(),
        })
    return rows

def resolve_cik(query: str, company_index: List[Dict]) -> Optional[int]:
    """Resolve user query to a CIK."""
    q = query.strip().upper()
    
    # exact ticker hit
    for row in company_index:
        if row["ticker"] and row["ticker"] == q:
            return row["cik"]

    # name contains
    contains = [r for r in company_index if q in r["name"]]
    if contains:
        return contains[0]["cik"]

    # name startswith
    starts = [r for r in company_index if r["name"].startswith(q)]
    if starts:
        return starts[0]["cik"]

    # token match
    tokens = set(re.findall(r"[A-Z0-9]+", q))
    best = None
    for r in company_index:
        name_tokens = set(r["name"].split())
        if tokens.issubset(name_tokens):
            best = r["cik"]
            break
    return best

# ---------- Enhanced Submissions Fetch ----------
def fetch_submissions_with_fallbacks(cik: int) -> Dict:
    """
    Get company submissions with fallback to historical data.
    The 'recent' endpoint only has ~2-3 years of filings.
    """
    url = SUBMISSIONS_URL_TMPL.format(cik=cik)
    print(f"📥 Fetching submissions from: {url}")
    
    with _session().get(url, timeout=30) as r:
        r.raise_for_status()
        data = r.json()
    
    # Debug info
    recent_filings = data.get("filings", {}).get("recent", {})
    forms = recent_filings.get("form", [])
    filing_dates = recent_filings.get("filingDate", [])
    
    print(f"📊 Found {len(forms)} recent filings")
    if forms:
        oldest_recent = filing_dates[-1] if filing_dates else "unknown"
        newest_recent = filing_dates[0] if filing_dates else "unknown"
        print(f"   Date range in recent filings: {oldest_recent} to {newest_recent}")
    
    form_counts = {}
    for form in forms:
        form_counts[form] = form_counts.get(form, 0) + 1
    
    for form, count in form_counts.items():
        print(f"   - {form}: {count} filings")
    
    return data

def _iterate_all_filings(filings_json: Dict, max_historical: int = 5) -> Iterable[Dict]:
    """
    Iterate through both recent filings and historical filings.
    The SEC API provides 'recent' filings and pointers to older archives.
    """
    # First, yield from recent filings
    recent = filings_json.get("filings", {}).get("recent", {})
    cols = ["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument"]
    arrays = [recent.get(k, []) for k in cols]
    
    if arrays and min(len(a) for a in arrays) > 0:
        for i in range(min(len(a) for a in arrays)):
            yield {k: arrays[j][i] for j, k in enumerate(cols)}
    
    # Then, try to fetch historical filings (older archives)
    archives = filings_json.get("filings", {}).get("files", [])
    print(f"📚 Found {len(archives)} historical archives")
    
    # Process most recent historical archives (oldest ones first in the list)
    for i, archive in enumerate(archives[:max_historical]):
        archive_name = archive.get("name")
        if archive_name:
            print(f"   Fetching historical archive: {archive_name}")
            try:
                archive_url = f"https://data.sec.gov/submissions/{archive_name}"
                time.sleep(0.2)  # Be polite
                
                with _session().get(archive_url, timeout=30) as r:
                    if r.status_code == 200:
                        archive_data = r.json()
                        archive_recent = archive_data.get("recent", {})
                        archive_arrays = [archive_recent.get(k, []) for k in cols]
                        
                        if archive_arrays and min(len(a) for a in archive_arrays) > 0:
                            count = min(len(a) for a in archive_arrays)
                            for j in range(count):
                                yield {k: archive_arrays[idx][j] for idx, k in enumerate(cols)}
                        print(f"   ✅ Added {count} filings from archive {i+1}")
                    else:
                        print(f"   ❌ Failed to fetch archive {i+1} (HTTP {r.status_code})")
            except Exception as e:
                print(f"   ⚠️  Error fetching archive {i+1}: {e}")

def build_primary_doc_url(cik: int | str, accession_no: str, primary_doc: str) -> str:
    """Build URL for primary document."""
    cik_no_zeros = str(int(cik))
    acc_nodash = accession_no.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_no_zeros}/{acc_nodash}/{primary_doc}"

def filings_in_range(
    filings_json: Dict,
    forms: Iterable[str],
    start: dt.date,
    end: dt.date,
    include_historical: bool = True,
) -> List[Dict]:
    """
    Filter filings by form type and filingDate within [start, end].
    Now includes historical archives for older filings.
    """
    hits = []
    forms_set = set(forms)
    
    print(f"🔎 Filtering for forms {forms_set} between {start} and {end}")
    if include_historical:
        print("📚 Including historical archives")
    
    # Choose iterator based on whether we want historical data
    if include_historical:
        filings_iter = _iterate_all_filings(filings_json)
    else:
        filings_iter = _iterate_recent(filings_json)
    
    for row in filings_iter:
        if row["form"] not in forms_set:
            continue
        try:
            fdate = dt.date.fromisoformat(row["filingDate"])
            if not (start <= fdate <= end):
                continue
                
            url = build_primary_doc_url(filings_json["cik"], row["accessionNumber"], row["primaryDocument"])
            hits.append({
                "form": row["form"],
                "filingDate": row["filingDate"],
                "reportDate": row.get("reportDate", ""),
                "accessionNumber": row["accessionNumber"],
                "primaryDocument": row["primaryDocument"],
                "url": url,
            })
        except (ValueError, KeyError) as e:
            continue
    
    print(f"✅ Found {len(hits)} matching filings")
    
    # Sort newest first
    hits.sort(key=lambda d: d["filingDate"], reverse=True)
    return hits

# ---------- Main Functions ----------
def find_10k_10q(
    company_or_ticker: str,
    year: Optional[int] = None,
    start: Optional[str] = None,
    end: Optional[str] = None,
    include_historical: bool = True,
) -> List[Dict]:
    """
    Enhanced main function that can find older filings using historical archives.
    """
    print(f"🚀 Starting SEC filings search for: {company_or_ticker}")
    
    idx = load_company_index()
    cik = resolve_cik(company_or_ticker, idx)
    if cik is None:
        raise ValueError(f"Could not resolve CIK for '{company_or_ticker}'. Try exact ticker or official EDGAR name.")

    # Be polite to SEC servers
    time.sleep(0.5)
    
    try:
        subs = fetch_submissions_with_fallbacks(cik)
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            raise ValueError(f"CIK {cik} not found in SEC database.")
        else:
            raise

    if year is not None:
        start_date = dt.date(year, 1, 1)
        end_date   = dt.date(year, 12, 31)
        print(f"📅 Searching year: {year}")
    else:
        if not start or not end:
            raise ValueError("Provide either 'year' OR both 'start' and 'end' (YYYY-MM-DD).")
        start_date = dt.date.fromisoformat(start)
        end_date   = dt.date.fromisoformat(end)
        print(f"📅 Searching range: {start} to {end}")

    results = filings_in_range(subs, {"10-K", "10-Q"}, start_date, end_date, include_historical)
    
    if not results:
        print("\n❌ No filings found. Suggestions:")
        print("   - Try setting include_historical=True (if not already)")
        print("   - The company may have used a different ticker/name in that year")
        print("   - Check if the company was public during that period")
        print("   - Try the direct EDGAR search: https://www.sec.gov/edgar/searchedgar/companysearch")
    
    return results

# ---------- Direct URL Fallback ----------
def get_direct_edgar_url(cik: str, year: int) -> str:
    """Generate direct EDGAR search URL as fallback."""
    return f"https://www.sec.gov/edgar/browse/?CIK={cik}&owner=exclude"

# ---------- Test Function ----------
def test_meta_2021():
    """Test specifically for META 2021 filings."""
    print("Testing META 2021 filings...")
    
    # META was Facebook in 2021, so try both
    for company in ["META", "FB", "FACEBOOK"]:
        try:
            print(f"\nTrying: {company}")
            idx = load_company_index()
            cik = resolve_cik(company, idx)
            if cik:
                print(f"✅ Found CIK: {cik} for {company}")
                results = find_10k_10q(company, year=2021, include_historical=True)
                if results:
                    print(f"🎉 Success! Found {len(results)} filings for {company} in 2021:")
                    for r in results:
                        print(f"   {r['form']} filed {r['filingDate']}: {r['url']}")
                    return results
        except Exception as e:
            print(f"❌ Failed with {company}: {e}")
    
    print("\n💡 Manual solution:")
    print("   META (Facebook) 2021 filings are available at:")
    print("   https://www.sec.gov/edgar/browse/?CIK=1326801&owner=exclude")
    print("   Filter for 10-K and 10-Q forms in 2021")
    return []

if __name__ == "__main__":
    # Test the enhanced function
    results = test_meta_2021()
    
    if not results:
        print("\n🔧 Alternative: Testing current year to verify basic functionality...")
        try:
            current_year = dt.datetime.now().year - 1  # Use last year to ensure filings exist
            results = find_10k_10q("META", year=current_year, include_historical=True)
            if results:
                print(f"✅ Basic functionality works! Found {len(results)} recent filings.")
        except Exception as e:
            print(f"❌ Basic test failed: {e}")

Testing META 2021 filings...

Trying: META
✅ Found CIK: 1326801 for META
🚀 Starting SEC filings search for: META
📥 Fetching submissions from: https://data.sec.gov/submissions/CIK0001326801.json
📊 Found 1001 recent filings
   Date range in recent filings: 2024-01-22 to 2025-10-07
   - 144: 385 filings
   - 4: 543 filings
   - 10-Q: 5 filings
   - 8-K: 15 filings
   - S-8 POS: 8 filings
   - SD: 2 filings
   - PX14A6G: 19 filings
   - 3: 7 filings
   - ARS: 2 filings
   - DEFA14A: 2 filings
   - DEF 14A: 2 filings
   - SCHEDULE 13G/A: 1 filings
   - 10-K: 2 filings
   - 424B2: 2 filings
   - FWP: 1 filings
   - PRE 14A: 1 filings
   - SC 13G/A: 4 filings
📅 Searching year: 2021
🔎 Filtering for forms {'10-Q', '10-K'} between 2021-01-01 and 2021-12-31
📚 Including historical archives
📚 Found 2 historical archives
   Fetching historical archive: CIK0001326801-submissions-001.json
   ⚠️  Error fetching archive 1: cannot access local variable 'count' where it is not associated with a value
   F